In [ ]:
# based on : https://www.tensorflow.org/tutorials/keras/classification


# Import tensorflow, this is a library which contains a lot of support for machine learning
import tensorflow as tf

# Numpy is a mathematics library which allows us to do manipulation of numerical data
import numpy as np

# Matplotlib is a library for plotting which also allows us to look at images
import matplotlib.pyplot as plt

In [ ]:
# Get the data sets

# Unlike the decision tree example the fashion mnist data set is loadable directly
# using a tensorflow function.
# Note that the first time you run this cell it will download the dataset from
# a store on the internet.

fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [ ]:
# Look at the data

# First how big are the datasets we just downloaded.
# note these are not pandas data frames they are just arrays of data.

print(f"The trining images are {train_images.shape} in size")
print(f"This means we have {train_images.shape[0]} images each of {train_images.shape[1]}x{train_images.shape[2]} pixels")

In [ ]:
print(train_labels.shape)

In [ ]:
# The labels we are given are simply class numbers. This is because of the way we are going to train our model.
# So here is a little lookup list of the class names.
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
img_num = 0
print(f"Image {img_num} of a image of class {train_labels[img_num]} which is a {class_names[train_labels[img_num]]}")

In [ ]:
# Let's take a look at the image itself

plt.figure()
plt.imshow(train_images[img_num])
plt.colorbar()
plt.grid(False)
plt.show()

In [ ]:
# Let's have a look at the numbers inside the image
# The following function shows us a small patch in the top left corner of the image
# which is 10 x 10 pixels.

print(train_images[img_num][0:10][0:10])

In [ ]:
# Note that the information is integers in the range 0 to 255.
# Unfortunately this is not what we want for our model. We want all inputs to be in the range 0 to 1.
# that's easily solved

train_images = train_images/255.0
test_images = test_images/255.0


In [ ]:
# now let's take another look

print(train_images[img_num][0:10][0:10])

In [ ]:
# Let's take a look at the image itself
# notice the scale on the right of the image.

plt.figure()
plt.imshow(train_images[img_num])
plt.colorbar()
plt.grid(False)
plt.show()

In [ ]:
# it's more sensible to look at grayscale images however so let's do that.
# We achieve this using the cmap option on the imshow function
# we also don't need the colour bar anymore so we'll drop that.

plt.figure()
plt.imshow(train_images[img_num], cmap=plt.cm.binary)
plt.grid(False)
plt.show()

In [ ]:
# It's all well and good looking at one image at a time, but let's try looking at a few
# This code uses the subplot functionality to put lots of images in a grid

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

In [ ]:
# OK so now we have all the data we need in a state where we can use it let's start building our model.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10)
])


model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# so what have we built here?
# Let's have a look at the model structure

model.summary()

In [ ]:
# Now we have the structure of our model. Let's train it.
# once again this is made trivial by the python libraries
#
# There is quite a lot going on under the hood however, so this will take a little time.

fit_data = model.fit(train_images, train_labels, epochs=10)

In [ ]:
# we asked the fitting function to store its history and now we can look at how
# it did at each epoch.

print(fit_data.history.keys())
plt.plot(fit_data.history['accuracy'])

In [ ]:
# Having got our model we can now look at it performs on the test images.

# We can use the predict function, like we did with the decision tree.
# again we will pass in the whole set of test images.
model.predict(test_images)

In [ ]:
# it's a bit difficult to see what we are being given here, so let's only
# send the model one image
img = test_images[0:1]
model.predict([img])

In [ ]:
# Why have we got 10 numbers returned, not just the number of the class?

print(f"The model thinks that the most likely class is {np.argmax(model.predict([img]))}")


In [ ]:
# as we did with the decision tree we can create a confusion matrix.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn import metrics
y_pred = model.predict(test_images)
pred_label = [np.argmax(i) for i in y_pred]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix(test_labels, pred_label))
cm_display.plot()
plt.show()
print(class_names)

In [ ]:
# Let's have have a go with a 'bigger' model
# And have a look at an interesting plugin called Tensor Board

# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime, os

# let's also make the program a little more structured using functions
# I've also allocated friendly names to my layers
def create_model():
  return tf.keras.Sequential([
      tf.keras.layers.Flatten(input_shape=(28, 28), name="input"),
      tf.keras.layers.Dense(300, activation='relu', name = "hidden_1"),
      tf.keras.layers.Dense(300, activation='relu', name="hidden_2"),
      tf.keras.layers.Dense(10, activation="softmax", name="output")
  ])

def train_model():
  model = create_model()
  model.compile(optimizer='sgd',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

  # give a place for training logs to be stored
  logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
  tensorboard_callback = tf.keras.callbacks.TensorBoard(logdir, histogram_freq=1)

  # now we will train the model. Notice this time I provide some validation (test) data
  # I also tell it to call the tensorboard logging function.
  model.fit(x=train_images,
            y=train_labels,
            epochs=30,
            validation_data=(test_images, test_labels),
            callbacks=[tensorboard_callback])

  return model

In [ ]:
# Let's start the tensorboard plug in
%tensorboard --logdir logs

In [ ]:
# now let's call our model training funciton
bigModel = train_model()

In [ ]:

# evaluate this new model and compare the with old model.
# compare their confusion matrices.

# is there any class which is better or worse with this new model.
# which samples were misclassified by one model but not the other?
#
